# Tutorial 3: Spec Contracts and Debugging

Estimated time: 25-35 minutes

## Prerequisites
No optional dependencies required.

## Learning aims
- Primary package aim: debug validation failures and restore a valid spec
- Secondary scientific aim: understand why explicit contracts protect scientific assumptions

## Success criteria
- you can intentionally break and repair a spec using validator output


## Why this tutorial matters

**Specs are contracts.** When a spec passes `bayesmm validate`, the framework guarantees a set of things hold: input names match adapter mappings, types are coercible, DOE values lie within declared support ranges, the storage root is project-relative, and so on. When you trust validation, you stop debugging by `print()` — the validator already told you exactly what's wrong, in a way Python's runtime errors never will.

This tutorial breaks two specs on purpose so you see the two error families you'll meet in practice: a **structural error** (a required section missing) and an **input-name mismatch** (the contract layers disagreeing about what something is called). The first is rare and obvious; the second is common and pernicious. By the end you'll be reading validator output as a partner, not a punishment.


## Step 1: Create a temporary copy and inject an error


In [1]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)


Repo root: /Users/barakraveh/Git/metamodeler_codex_scaffold_docs


In [2]:
import json
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

src = root / 'tutorials/specs/model.toy.grid.json'
dst = root / 'tmp/tutorials_modelspec_edit.json'
dst.parent.mkdir(parents=True, exist_ok=True)
dst.write_text(src.read_text())

payload = json.loads(dst.read_text())
payload['design'].pop('grid', None)
dst.write_text(json.dumps(payload, indent=2))
print('Wrote broken spec:', dst)

Wrote broken spec: /Users/barakraveh/Git/metamodeler_codex_scaffold_docs/tmp/tutorials_modelspec_edit.json


## Step 2: Validate and inspect actionable error


In [3]:
run_mm_cli('validate', 'tmp/tutorials_modelspec_edit.json', check=False)


$ bayesmm validate tmp/tutorials_modelspec_edit.json
Spec validation failed:
- design: Value error, design.grid is required when design.strategy is 'grid'


1

**Expected output:** The validator should report a validation error because the `grid` key was removed from the `design` section. You should see a message like:

```
Spec validation failed: 1 validation error for ModelSpec
...
  grid config required when strategy is 'grid'
```

This actionable error tells you exactly which section is broken and what contract was violated.

## Step 3: Repair and re-validate


In [4]:
import json
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

p = root / 'tmp/tutorials_modelspec_edit.json'
payload = json.loads(p.read_text())
payload['design']['grid'] = {'a': [0.0, 1.0, 2.0], 'b': [0.0, 1.0, 2.0]}
p.write_text(json.dumps(payload, indent=2))
print('Spec repaired')

Spec repaired


**Expected output:** After repair, the validator should print a success message:

```
Spec is valid.  (model='test', strategy=grid, 9 design points)
```

If you still see errors, double-check that the `grid` key maps variable names to lists of values matching the `io_schema.inputs` names.

In [5]:
run_mm_cli('validate', 'tmp/tutorials_modelspec_edit.json')


$ bayesmm validate tmp/tutorials_modelspec_edit.json
Spec validation passed: model=toy_program_tutorial, strategy=grid, adapter=python_cli_adapter_v1


0

## Step 4: A second error type — per-input contract violation

The first cycle (`grid` removed) was a **structural** error: a required field missing. The error type below is a **semantic** one: the spec is structurally fine, but a value violates a per-input contract you'd want enforced. Specifically, we'll invert a `support: [min, max]` range to `[max, min]`. Easy typo to make. The DOE planner would happily run it; the validator stops it before any model executes.


In [6]:
import json
from pathlib import Path

p = root / "tmp/tutorials_modelspec_edit.json"
payload = json.loads(p.read_text())

# Inject the bug: invert the support range on the first input from [0.0, 2.0]
# to [2.0, 0.0]. The list still has two numbers, so it's structurally a
# valid `support` field — but min > max is contractually broken.
payload["io_schema"]["inputs"][0]["support"] = [2.0, 0.0]
p.write_text(json.dumps(payload, indent=2))

print("Wrote spec with inverted support:", p)
print(
    "io_schema.inputs[0]:",
    payload["io_schema"]["inputs"][0]["name"],
    "support =",
    payload["io_schema"]["inputs"][0]["support"],
)


Wrote spec with inverted support: /Users/barakraveh/Git/metamodeler_codex_scaffold_docs/tmp/tutorials_modelspec_edit.json
io_schema.inputs[0]: a support = [2.0, 0.0]


In [7]:
run_mm_cli("validate", "tmp/tutorials_modelspec_edit.json", check=False)


$ bayesmm validate tmp/tutorials_modelspec_edit.json
Spec validation failed:
- io_schema.inputs.0.support: Value error, support max must be greater than support min


1

**Expected output:** The validator rejects the spec with a per-input contract violation, naming the exact field path. You should see:

```
Spec validation failed:
- io_schema.inputs.0.support: Value error, support max must be greater than support min
```

Read the path before reading the message: `io_schema.inputs.0.support` is "the `support` field of the first entry of `io_schema.inputs`" — the validator points you at the bad field directly. Real validation errors have this same shape across the framework.


In [8]:
import json
from pathlib import Path

p = root / "tmp/tutorials_modelspec_edit.json"
payload = json.loads(p.read_text())

# Repair: swap the bounds back to (min, max).
payload["io_schema"]["inputs"][0]["support"] = [0.0, 2.0]
p.write_text(json.dumps(payload, indent=2))

run_mm_cli("validate", "tmp/tutorials_modelspec_edit.json")


$ bayesmm validate tmp/tutorials_modelspec_edit.json
Spec validation passed: model=toy_program_tutorial, strategy=grid, adapter=python_cli_adapter_v1


0

## Scientific checkpoint

For each input variable in your own future model, plan ahead the spec contract you'd want the validator to enforce:

- **Name and units.** Match the `io_schema.inputs[*].name` exactly to whatever you'll reference in `design.grid` and `adapter.input_mapping`. Mismatches are the most common spec bug (see Step 4 above).
- **Type and support range.** What's the physical meaning? What range covers the regime you actually care about? `support: [min, max]` is what the DOE planner uses to clip values; if you set it wrong, the framework will dutifully run nonsense without complaining.
- **Anticipated bias.** What scientific bias could appear if your support is too narrow (cherry-picked regime) or too wide (extrapolation outside model validity)?

The validator catches contract violations. It does NOT catch *scientific* mistakes — that's still on you. The contract just means your scientific reasoning happens once, in the spec, and is then enforced consistently across every run that uses it.


## Spec sections as contracts

Each top-level section of a `ModelSpec` is a contract layer. When `bayesmm validate` says a spec is valid, it's guaranteeing the things in the right column hold *for every cell, every CLI invocation, every downstream tool* that consumes the spec. The validator is the boundary between "you typed this" and "the framework treats it as truth."

| Section | What it contracts for |
|---|---|
| `model` | Identity (name, version) and entrypoint (`type: local` + `entrypoint: [...]`, or `type: biomodels` + `biomodels_id`). The framework knows what to run and what to call it. |
| `runner` | Where and how to execute (`mode: local_process | mpi | ...`), declared resources (`cpus`, `mem_gb`, `walltime_min`), sweep mode, and optional environment (`execution_env.conda_env`). |
| `io_schema` | The variable contract: typed `inputs` and `outputs` with names, types, units, and (for floats) `support` ranges. Every other section that names a variable is checked against this list. |
| `design` | The DOE contract: `strategy: grid | sobol`, plus the per-strategy parameters (`grid` dict, or `sobol.n_points`/`seed`). The variable names used here MUST appear in `io_schema.inputs`. |
| `adapter` | How to translate a DOE point into a process invocation. `id: python_cli_adapter_v1` (or `biomodels_sbml_adapter_v1`), with `input_mapping` and `output_mapping` describing how spec variables flow into and out of the executable. |
| `reproducibility` | The `seed` (and any related provenance fields). Required for any deterministic re-run guarantee. |
| `storage` | Where artifacts go: `root` (project-relative path; never absolute, never `..`-traversal). The framework writes the centralized sweep CSV, the run registry, and surrogate artifacts under here. |

When validation fails, the error message names the section and the violated rule. Read it. The error is the *contract violation*, not a mystery — it's pointing at exactly the layer that's broken.
